# 03 · Option 2 · Azure AI Foundry Evaluations

The harness captures Copilot Chat answers and Workspace Monitoring DAX traces once, then scores each ask through the **Foundry evaluation backend** (`option="foundry"` or `option="both"`):

* **Code evaluators:** deterministic `value`, `artifact`, and `model` checks.
* **Prompt & built-in evaluators:** explanation consistency, coherence, and similarity.

Defaults safely to `MOCK = True` (using `LocalFoundryEmulator` offline) whenever live Foundry or Copilot credentials are absent.

In [ ]:
import pathlib
import sys

repo = pathlib.Path.cwd()
while repo != repo.parent and not (repo / "src" / "cfeval").exists():
    repo = repo.parent
if (repo / "src" / "cfeval").exists() and str(repo / "src") not in sys.path:
    sys.path.insert(0, str(repo / "src"))

import cfeval
from cfeval.config import Settings

settings = Settings.load(env_file=str(repo / ".env"), output_dir=str(repo / "build" / "notebook-runs"))
has_live_foundry = bool(
    settings.tenant_id and settings.client_id and settings.test_user_upn and settings.foundry_project_endpoint
)
MOCK = True if not has_live_foundry else True  # Set MOCK = False explicitly to submit a live Foundry evaluation
print(f"cfeval {cfeval.__version__} | has_live_foundry={has_live_foundry} | MOCK={MOCK}")

## Run Option 1 + Option 2 Side by Side

When `MOCK = True`, `LocalFoundryEmulator` exercises the full Option 2 normalization path offline without network calls.

In [ ]:
from cfeval.cli import build_harness

outcome = build_harness(
    settings,
    option="both",
    mock=MOCK,
    publish=False,
    profile="tuned",
    run_id="nb03-foundry-compare",
    run_name="Option 1 vs Option 2 Comparison",
).run()

print(f"Run {outcome.run_id} | status={outcome.status} | metrics={outcome.metrics}")

In [ ]:
cols = ["QuestionId", "Backend", "Verdict", "ValueScore", "ArtifactScore", "ModelScore"]
for row in outcome.results[:16]:
    print({k: row.get(k) for k in cols})

## What Foundry Stores in Live Mode

* Versioned code evaluators (`cfeval_value_match`, `cfeval_artifact_match`, `cfeval_model_match`) and prompt/built-in evaluators.
* One Foundry evaluation run linked via `EvalRun.FoundryReportUrl`.